# Linear Regression from Scratch: Clinical Dosing Examples

This notebook builds linear regression up from first principles with PyTorch, using two
clinical relationships that really are close to linear:

1. **Body weight (kg) → total daily insulin requirement (units/day)** — the one-variable case.
   Insulin in type 1 diabetes is dosed per kilogram, roughly 0.5–0.6 units/kg/day.
2. **Dose (mg) + renal function (eGFR) → peak plasma concentration (mg/L)** — the two-variable
   case, fitted as a plane. More drug raises the peak; better kidney function clears the drug
   faster and lowers it.

You will see the same model solved three ways: animated gradient descent, the loss surface the
optimizer is walking down, and the exact normal-equation solution.

> **A note on the data.** Every patient in this notebook is simulated from published
> clinical ranges. No real or identifiable patient data is used, and none of the fitted
> models should be used for actual dosing.

---


In [1]:
import numpy as np
import plotly.graph_objs as go
from plotly.subplots import make_subplots
from IPython.display import display, clear_output


In [2]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import networkx as nx
import numpy as np

# Option 1: Simple Matplotlib visualization
def visualize_network_matplotlib(model):
    """
    Create a simple network diagram using matplotlib
    """
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Network structure
    layers = [1, 100, 1]  # input, hidden, output
    layer_names = ['Input', 'Hidden (100)', 'Output']
    
    # Position nodes
    max_nodes = max(layers)
    positions = {}
    node_id = 0
    
    for layer_idx, num_nodes in enumerate(layers):
        x = layer_idx * 3
        start_y = (max_nodes - num_nodes) / 2
        
        for node_idx in range(min(num_nodes, 10)):  # Show max 10 nodes per layer
            y = start_y + node_idx * (max_nodes / max(num_nodes, 10))
            positions[node_id] = (x, y)
            
            # Draw node
            circle = patches.Circle((x, y), 0.2, facecolor='lightblue', 
                                  edgecolor='black', linewidth=1)
            ax.add_patch(circle)
            node_id += 1
        
        # Add layer label
        ax.text(x, -1, layer_names[layer_idx], ha='center', fontsize=12, fontweight='bold')
        
        # Add "..." if layer has more than 10 nodes
        if num_nodes > 10:
            ax.text(x, start_y + 10, '...', ha='center', fontsize=16, fontweight='bold')
    
    # Draw connections (sample)
    for layer_idx in range(len(layers) - 1):
        start_layer_size = min(layers[layer_idx], 10)
        end_layer_size = min(layers[layer_idx + 1], 10)
        
        for i in range(start_layer_size):
            for j in range(min(end_layer_size, 5)):  # Limit connections for clarity
                start_pos = (layer_idx * 3, (max_nodes - min(layers[layer_idx], 10)) / 2 + i * (max_nodes / max(min(layers[layer_idx], 10), 10)))
                end_pos = ((layer_idx + 1) * 3, (max_nodes - min(layers[layer_idx + 1], 10)) / 2 + j * (max_nodes / max(min(layers[layer_idx + 1], 10), 10)))
                
                ax.plot([start_pos[0] + 0.2, end_pos[0] - 0.2], 
                       [start_pos[1], end_pos[1]], 'k-', alpha=0.3, linewidth=0.5)
    
    ax.set_xlim(-0.5, (len(layers) - 1) * 3 + 0.5)
    ax.set_ylim(-2, max_nodes + 1)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title('Neural Network Architecture', fontsize=16, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

# Option 2: NetworkX visualization with weight visualization
def visualize_network_with_weights(model):
    """
    Create a network visualization showing connection weights
    """
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # Get weights
    weights1 = model.fc1.weight.data.numpy()  # 100x1
    weights2 = model.fc2.weight.data.numpy()  # 1x100
    
    # Create simplified network (show only subset of hidden nodes)
    num_hidden_show = 10
    
    # Create graph
    G = nx.Graph()
    
    # Add nodes
    G.add_node("Input", pos=(0, 0), layer=0)
    
    for i in range(num_hidden_show):
        G.add_node(f"H{i}", pos=(2, i - num_hidden_show/2), layer=1)
    
    G.add_node("Output", pos=(4, 0), layer=2)
    
    # Add edges with weights
    # Input to hidden
    for i in range(num_hidden_show):
        weight = weights1[i, 0]
        G.add_edge("Input", f"H{i}", weight=weight)
    
    # Hidden to output
    for i in range(num_hidden_show):
        weight = weights2[0, i]
        G.add_edge(f"H{i}", "Output", weight=weight)
    
    # Get positions
    pos = nx.get_node_attributes(G, 'pos')
    
    # Draw nodes
    node_colors = ['red' if 'Input' in node else 'blue' if 'Output' in node else 'lightgreen' for node in G.nodes()]
    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=500, ax=ax)
    nx.draw_networkx_labels(G, pos, ax=ax)
    
    # Draw edges with weights as colors
    edges = G.edges()
    weights = [G[u][v]['weight'] for u, v in edges]
    
    # Normalize weights for color mapping
    max_weight = max(abs(w) for w in weights)
    normalized_weights = [w / max_weight for w in weights]
    
    edge_colors = ['red' if w < 0 else 'blue' for w in normalized_weights]
    edge_widths = [abs(w) * 3 for w in normalized_weights]
    
    nx.draw_networkx_edges(G, pos, edge_color=edge_colors, width=edge_widths, alpha=0.6, ax=ax)
    
    ax.set_title('Neural Network with Connection Weights\n(Red=Negative, Blue=Positive, Width=Magnitude)', 
                fontsize=14, fontweight='bold')
    ax.axis('off')
    
    plt.tight_layout()
    plt.show()

# Option 3: Plotly interactive network visualization
def visualize_network_plotly_interactive(model):
    """
    Create an interactive network visualization using Plotly
    """
    import plotly.graph_objects as go
    import plotly.express as px
    
    # Get a subset of weights for visualization
    weights1 = model.fc1.weight.data.numpy()[:12, :]  # Show first 20 hidden neurons
    weights2 = model.fc2.weight.data.numpy()[:, :12]  # Corresponding output weights
    
    # Create node positions
    input_pos = (0, 0)
    hidden_pos = [(1, i - 10) for i in range(212)]
    output_pos = (2, 0)
    
    # All positions
    all_pos = [input_pos] + hidden_pos + [output_pos]
    
    # Create edge traces
    edge_x = []
    edge_y = []
    edge_colors = []
    edge_widths = []
    
    # Input to hidden connections
    for i in range(12):
        weight = weights1[i, 0]
        edge_x.extend([input_pos[0], hidden_pos[i][0], None])
        edge_y.extend([input_pos[1], hidden_pos[i][1], None])
        edge_colors.append('red' if weight < 0 else 'blue')
        edge_widths.append(abs(weight) * 2)
    
    # Hidden to output connections
    for i in range(20):
        weight = weights2[0, i]
        edge_x.extend([hidden_pos[i][0], output_pos[0], None])
        edge_y.extend([hidden_pos[i][1], output_pos[1], None])
        edge_colors.append('red' if weight < 0 else 'blue')
        edge_widths.append(abs(weight) * 2)
    
    # Create edge trace
    edge_trace = go.Scatter(x=edge_x, y=edge_y,
                           line=dict(width=1, color='gray'),
                           hoverinfo='none',
                           mode='lines')
    
    # Create node traces
    node_x = [pos[0] for pos in all_pos]
    node_y = [pos[1] for pos in all_pos]
    
    node_trace = go.Scatter(x=node_x, y=node_y,
                           mode='markers+text',
                           hoverinfo='text',
                           text=['Input'] + [f'H{i}' for i in range(20)] + ['Output'],
                           textposition="middle center",
                           marker=dict(showscale=True,
                                     colorscale='RdYlBu',
                                     reversescale=True,
                                     color=[],
                                     size=30,
                                     colorbar=dict(thickness=15,
                                                 xanchor="left",
                                                 titleside="right"
                                                ),
                                     line_width=2))
    
    # Color nodes by their average connection weight
    node_colors = [0]  # Input node
    for i in range(12):
        avg_weight = (weights1[i, 0] + weights2[0, i]) / 2
        node_colors.append(avg_weight)
    node_colors.append(np.mean(weights2[0, :20]))  # Output node
    
    node_trace.marker.color = node_colors
    
    fig = go.Figure(data=[edge_trace, node_trace],
                   layout=go.Layout(
                        title='Interactive Neural Network Visualization',
                        titlefont_size=16,
                        showlegend=False,
                        hovermode='closest',
                        margin=dict(b=20,l=5,r=5,t=40),
                        annotations=[ dict(
                            text="Node colors represent average connection weights",
                            showarrow=False,
                            xref="paper", yref="paper",
                            x=0.005, y=-0.002,
                            xanchor='left', yanchor='bottom',
                            font=dict(size=12)
                        )],
                        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False)))
    
    fig.show()

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import clear_output
import time

# --- TOGGLE LOGIC ---
# Run this cell multiple times to toggle between states
# First run: shows all parameters as 0
# Second run: performs gradient descent
try:
    run_linear_regression
except NameError:
    run_linear_regression = False

# Flip the switch
run_linear_regression = not run_linear_regression

# 1. Generate Synthetic Data - Clinical Example: Body Weight vs Daily Insulin Requirement
# Total daily insulin dose in type 1 diabetes is dosed per kilogram:
# roughly 0.5-0.6 units/kg/day, which makes it a genuinely near-linear
# clinical relationship. All values below are SIMULATED - no real patient data.
np.random.seed(42)
# Body weight in kilograms (range: 40 to 120 kg)
weight_numpy = np.linspace(40, 120, 100)
# True function: insulin = 0.55 * weight + 2
# A 60 kg adult needs ~35 units/day, a 100 kg adult ~57 units/day
insulin_numpy = 0.55 * weight_numpy + 2 + np.random.normal(0, 3, size=weight_numpy.shape)

weight_tensor = torch.from_numpy(weight_numpy.astype(np.float32)).view(-1, 1)
insulin_tensor = torch.from_numpy(insulin_numpy.astype(np.float32)).view(-1, 1)

# 2. Define Model
class LinearRegressionModel(nn.Module):
    def __init__(self):
        super(LinearRegressionModel, self).__init__()
        self.linear = nn.Linear(1, 1)

    def forward(self, x):
        return self.linear(x)

# Helper for visualization
def flatten_parameters(model):
    params = []
    for param in model.parameters():
        params.extend(param.view(-1).tolist())
    return params

# Helper to create the figure (reused in both modes)
def create_figure(weight, insulin, predicted, current_params, losses, title, fit_color='red'):
    # Params are [slope (units per kg), intercept (units/day)]
    slope_display = current_params[0]      # insulin units per kg of body weight
    intercept_display = current_params[1]  # units/day at zero weight

    trace_data = go.Scatter(x=weight, y=insulin, mode='markers', name='Data')
    trace_fit = go.Scatter(
        x=weight, y=predicted, mode='lines', name='Fit',
        line=dict(color=fit_color, width=3)
    )

    display_params = [slope_display, intercept_display]
    colors = ['green' if val >= 0 else 'red' for val in display_params] if fit_color != 'gray' else ['gray', 'gray']
    trace_parameters = go.Bar(
        x=['Slope (units/kg)', 'Intercept (units)'],
        y=display_params,
        name='Parameters',
        marker_color=colors,
        text=[f'{val:.2f}' for val in display_params],
        textposition='outside',
        textfont=dict(size=12)
    )

    trace_loss = go.Scatter(
        x=list(range(len(losses))) if losses else [0],
        y=losses if losses else [0],
        mode='lines', name='Loss',
        line=dict(color='blue')
    )

    fig = make_subplots(
        rows=2, cols=2,
        column_widths=[0.7, 0.3],
        row_heights=[0.6, 0.4],
        subplot_titles=('Daily Insulin Requirement vs Body Weight', 'Parameters', 'Loss'),
        specs=[[{"rowspan": 2}, {}], [None, {}]]
    )

    fig.add_trace(trace_data, row=1, col=1)
    fig.add_trace(trace_fit, row=1, col=1)
    fig.add_trace(trace_parameters, row=1, col=2)
    fig.add_trace(trace_loss, row=2, col=2)

    fig.update_xaxes(title_text="Body Weight (kg)", row=1, col=1)
    fig.update_yaxes(title_text="Daily Insulin (units/day)", row=1, col=1)

    fig.update_yaxes(range=[0, 80], row=1, col=1)
    fig.update_yaxes(range=[-1, 5], row=1, col=2)
    if not losses:
        fig.update_yaxes(range=[0, 2500], row=2, col=2)
    fig.update_layout(title=title, width=1000, height=600, showlegend=False)

    return fig

# Create model and initialize to zero
model = LinearRegressionModel()
with torch.no_grad():
    model.linear.weight.fill_(0.0)
    model.linear.bias.fill_(0.0)

# --- EXECUTION MODES ---

if not run_linear_regression:
    # === MODE A: RESET TO ZERO (STATIC) ===

    with torch.no_grad():
        predicted = model(weight_tensor).numpy().flatten()
    current_params = flatten_parameters(model)

    fig = create_figure(
        weight_numpy, insulin_numpy, predicted, current_params, [],
        'State: RESET (Initialized to 0) - Run again to train',
        fit_color='red'
    )
    fig.show()

else:
    # === MODE B: START REGRESSION (ANIMATION) ===

    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.05)

    num_epochs = 2000
    losses = []

    # Show initial state BEFORE any training (epoch 0)
    with torch.no_grad():
        predicted = model(weight_tensor).numpy().flatten()
    current_params = flatten_parameters(model)

    fig = create_figure(
        weight_numpy, insulin_numpy, predicted, current_params, [],
        'State: LEARNING | Epoch 0 | Starting...',
        fit_color='red'
    )
    fig.show()
    time.sleep(0.5)

    # 3. Training Loop
    for epoch in range(num_epochs):
        optimizer.zero_grad()
        outputs = model(weight_tensor)
        loss = criterion(outputs, insulin_tensor)
        loss.backward()
        optimizer.step()

        losses.append(loss.item())

        update_freq = 1 if epoch < 20 else 20
        if (epoch + 1) % update_freq == 0:
            clear_output(wait=True)

            with torch.no_grad():
                predicted = model(weight_tensor).numpy().flatten()
            current_params = flatten_parameters(model)

            fig = create_figure(
                weight_numpy, insulin_numpy, predicted, current_params, losses,
                f'State: LEARNING | Epoch {epoch+1}/{num_epochs} | Loss: {loss.item():.4f}',
                fit_color='red'
            )
            fig.show()
            time.sleep(0.075)


In [4]:
# Gradient Descent - Clinical Example: Body Weight vs Daily Insulin Requirement
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import plotly.graph_objects as go
from IPython.display import clear_output
import time

# --- TOGGLE LOGIC ---
try:
    run_gradient_descent
except NameError:
    run_gradient_descent = False

run_gradient_descent = not run_gradient_descent

# --- 1. SETUP DATA ---
# Clinical Example: Body Weight (kg) vs Total Daily Insulin Dose (units/day)
# True relationship: insulin = 0.55 * weight + 2   (simulated data)
np.random.seed(42)
weight_numpy = np.linspace(40, 120, 100)
insulin_numpy = 0.55 * weight_numpy + 2 + np.random.normal(0, 3, size=weight_numpy.shape)

weight_tensor = torch.from_numpy(weight_numpy.astype(np.float32)).view(-1, 1)
insulin_tensor = torch.from_numpy(insulin_numpy.astype(np.float32)).view(-1, 1)

# --- 2. PRE-CALCULATE LANDSCAPE ---
# Slope: insulin units per kg (true value ~0.55, the classic 0.5-0.6 units/kg rule)
# Intercept: units/day at zero weight (true value ~2)
w_values = np.linspace(0, 1.1, 40)
b_values = np.linspace(-6, 8, 40)
w_grid, b_grid = np.meshgrid(w_values, b_values)
loss_grid = np.zeros_like(w_grid)

for i in range(w_grid.shape[0]):
    for j in range(w_grid.shape[1]):
        pred = w_grid[i, j] * weight_numpy + b_grid[i, j]
        loss_grid[i, j] = np.mean((insulin_numpy - pred)**2)

# Initial position
initial_w, initial_b = 0.0, -4.0
initial_loss = np.mean((insulin_numpy - (initial_w * weight_numpy + initial_b))**2)

# --- CAMERA SETUP ---
camera_view = dict(eye=dict(x=0.3, y=1.5, z=1.2))

def create_figure(path_w, path_b, path_loss, title):
    fig = go.Figure()

    fig.add_trace(go.Surface(
        x=w_grid, y=b_grid, z=loss_grid,
        colorscale='Viridis', opacity=0.8,
        showscale=False, name='Landscape'
    ))

    fig.add_trace(go.Scatter3d(
        x=path_w, y=path_b, z=path_loss,
        mode='lines', line=dict(color='white', width=5),
        name='Path'
    ))

    fig.add_trace(go.Scatter3d(
        x=[path_w[-1]], y=[path_b[-1]], z=[path_loss[-1]],
        mode='markers', marker=dict(size=10, color='yellow'),
        name='Agent'
    ))

    fig.update_layout(
        title=title,
        scene=dict(
            xaxis_title='Slope (units per kg)',
            yaxis_title='Intercept (units/day)',
            zaxis_title='Loss (MSE)',
            camera=camera_view
        ),
        width=900, height=600,
        showlegend=False,
        margin=dict(l=0, r=0, b=0, t=60)
    )

    return fig

# --- EXECUTION MODES ---

if not run_gradient_descent:
    fig = create_figure(
        [initial_w], [initial_b], [initial_loss],
        'Gradient Descent (Body Weight → Daily Insulin) - Run again to start'
    )
    fig.show()

else:
    model = nn.Linear(1, 1)
    with torch.no_grad():
        model.weight.fill_(initial_w)
        model.bias.fill_(initial_b)

    optimizer = optim.Adam(model.parameters(), lr=0.1)
    criterion = nn.MSELoss()

    path_w = [initial_w]
    path_b = [initial_b]
    path_loss = [initial_loss]

    fig = create_figure(path_w, path_b, path_loss, 'Epoch 0 | Starting...')
    fig.show()
    time.sleep(1.0)

    for epoch in range(500):
        optimizer.zero_grad()
        outputs = model(weight_tensor)
        loss = criterion(outputs, insulin_tensor)
        loss.backward()
        optimizer.step()

        # Record AFTER step, but recalculate loss at new position
        w_curr = model.weight.item()
        b_curr = model.bias.item()
        # Recalculate loss to match the surface calculation exactly
        with torch.no_grad():
            l_curr = criterion(model(weight_tensor), insulin_tensor).item()

        path_w.append(w_curr)
        path_b.append(b_curr)
        path_loss.append(l_curr)

        if (epoch + 1) % 5 == 0:
            clear_output(wait=True)
            fig = create_figure(
                path_w, path_b, path_loss,
                f'Epoch {epoch+1} | Loss: {l_curr:.4f}'
            )
            fig.show()
            time.sleep(0.05)


In [5]:
# Calculated Linear Regression - Clinical Example: Body Weight vs Daily Insulin Requirement
import torch
import torch.nn as nn
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- 1. GLOBAL TOGGLE LOGIC ---
try:
    is_solved
except NameError:
    is_solved = False

is_solved = not is_solved

# --- 2. SETUP DATA & MODEL ---
# Clinical Example: Body Weight (kg) vs Total Daily Insulin Dose (units/day)
# Simulated data - no real patient records are used.
np.random.seed(42)
weight_numpy = np.linspace(40, 120, 100)
insulin_numpy = 0.55 * weight_numpy + 2 + np.random.normal(0, 3, size=weight_numpy.shape)

weight_tensor = torch.from_numpy(weight_numpy.astype(np.float32)).view(-1, 1)
insulin_tensor = torch.from_numpy(insulin_numpy.astype(np.float32)).view(-1, 1)

model = nn.Linear(1, 1)

# --- 3. APPLY LOGIC BASED ON TOGGLE ---
if is_solved:
    status_msg = "Solved"
    line_color = 'red'

    # Mathematical Solve (Normal Equation)
    ones = torch.ones_like(weight_tensor)
    X_design = torch.cat([weight_tensor, ones], dim=1)
    theta = torch.inverse(X_design.t() @ X_design) @ X_design.t() @ insulin_tensor

    slope_val = theta[0].item()
    intercept_val = theta[1].item()

    with torch.no_grad():
        model.weight.fill_(slope_val)
        model.bias.fill_(intercept_val)

else:
    status_msg = "Initial"
    line_color = 'red'

    slope_val = 0.0
    intercept_val = 0.0

    with torch.no_grad():
        model.weight.fill_(0.0)
        model.bias.fill_(0.0)

# --- 4. VISUALIZE ---
with torch.no_grad():
    predicted = model(weight_tensor).numpy().flatten()
    current_loss = torch.mean((insulin_tensor - model(weight_tensor))**2).item()

# Display slope as insulin units per kg of body weight
slope_display = slope_val
intercept_display = intercept_val

trace_data = go.Scatter(x=weight_numpy, y=insulin_numpy, mode='markers', name='Data', marker=dict(opacity=0.5))

trace_fit = go.Scatter(
    x=weight_numpy, y=predicted, mode='lines', name='Model Fit',
    line=dict(color=line_color, width=4)
)

colors = ['green' if val > 0.01 else 'gray' for val in [slope_display, intercept_display]]
trace_parameters = go.Bar(
    x=['Slope (units/kg)', 'Intercept (units)'],
    y=[slope_display, intercept_display],
    name='Params',
    marker_color=colors,
    text=[f'{slope_display:.2f}', f'{intercept_display:.2f}'],
    textposition='outside',
    textfont=dict(size=12)
)

fig = make_subplots(
    rows=1, cols=2, column_widths=[0.7, 0.3],
    subplot_titles=('Daily Insulin Requirement vs Body Weight', 'Parameters')
)

fig.add_trace(trace_data, row=1, col=1)
fig.add_trace(trace_fit, row=1, col=1)
fig.add_trace(trace_parameters, row=1, col=2)

fig.update_xaxes(title_text="Body Weight (kg)", row=1, col=1)
fig.update_yaxes(title_text="Daily Insulin (units/day)", row=1, col=1)

fig.update_yaxes(range=[0, 80], row=1, col=1)
fig.update_yaxes(range=[-1, 5], row=1, col=2)

fig.update_layout(
    title=f'State: {status_msg} | Loss: {current_loss:.4f}',
    width=1000, height=500, showlegend=False
)

fig.show()


In [6]:
# 3D Calculated Linear Regression (Plane Fit)
# Clinical Example: Predicting Peak Plasma Concentration from Dose and Renal Function
import torch
import torch.nn as nn
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- TOGGLE LOGIC ---
try:
    is_solved_3d
except NameError:
    is_solved_3d = False

is_solved_3d = not is_solved_3d

# --- 1. SETUP 3D DATA ---
# Simulated pharmacokinetic study - no real patient data.
np.random.seed(42)
n_points = 100

# Two input features (clinical context)
dose_mg = np.random.uniform(50, 500, n_points)   # Administered dose (mg)
egfr = np.random.uniform(15, 120, n_points)      # Renal function, eGFR (mL/min/1.73m^2)

# Rescale both predictors so the learned coefficients are on a comparable scale
# (dose per 100 mg, eGFR per 10 mL/min) - the same trick as standardizing inputs.
dose_100mg = dose_mg / 100.0
egfr_10 = egfr / 10.0

# True relationship: Cmax = 2.2*dose_100mg - 0.45*egfr_10 + 5 + noise
# More drug in = higher peak concentration.
# Better kidney function = faster clearance = LOWER peak concentration,
# which is why the eGFR coefficient is negative.
true_w1, true_w2, true_b = 2.2, -0.45, 5.0
peak_concentration = true_w1 * dose_100mg + true_w2 * egfr_10 + true_b + np.random.normal(0, 0.6, n_points)

# Convert to tensors
X_tensor = torch.from_numpy(np.column_stack([dose_100mg, egfr_10]).astype(np.float32))
z_tensor = torch.from_numpy(peak_concentration.astype(np.float32)).view(-1, 1)

# Define Model (2 inputs -> 1 output)
model_3d = nn.Linear(2, 1)

# --- 2. APPLY LOGIC BASED ON TOGGLE ---
if is_solved_3d:
    # === MODE: CALCULATE EXACT SOLUTION (Normal Equation) ===
    status_msg = "Solved"
    plane_color = 'gray'
    plane_opacity = 0.6

    # Add bias column
    ones = torch.ones(X_tensor.shape[0], 1)
    X_design = torch.cat([X_tensor, ones], dim=1)

    # Normal equation: theta = (X'X)^-1 X'y
    theta = torch.inverse(X_design.t() @ X_design) @ X_design.t() @ z_tensor

    w1_val = theta[0].item()
    w2_val = theta[1].item()
    b_val = theta[2].item()

    # Set model weights
    with torch.no_grad():
        model_3d.weight[0, 0] = w1_val
        model_3d.weight[0, 1] = w2_val
        model_3d.bias[0] = b_val

else:
    # === MODE: RESET TO ZERO ===
    status_msg = "Initial"
    plane_color = 'gray'
    plane_opacity = 0.4

    w1_val, w2_val, b_val = 0.0, 0.0, 0.0

    with torch.no_grad():
        model_3d.weight.fill_(0.0)
        model_3d.bias.fill_(0.0)

# --- 3. CALCULATE LOSS ---
with torch.no_grad():
    predictions = model_3d(X_tensor)
    current_loss = torch.mean((z_tensor - predictions)**2).item()

# --- 4. CREATE PLANE SURFACE FOR VISUALIZATION ---
# Grid in the clinician-facing units (mg and mL/min), plane evaluated on the rescaled inputs
dose_grid = np.linspace(50, 500, 20)
egfr_grid = np.linspace(15, 120, 20)
Dose_mesh, Egfr_mesh = np.meshgrid(dose_grid, egfr_grid)

# Calculate predicted peak concentration for the plane
Conc_plane = w1_val * (Dose_mesh / 100.0) + w2_val * (Egfr_mesh / 10.0) + b_val

# --- 5. VISUALIZE WITH PARAMETER BAR CHART ---
fig = make_subplots(
    rows=1, cols=2,
    column_widths=[0.75, 0.25],
    specs=[[{"type": "scene"}, {"type": "xy"}]],
    subplot_titles=('Peak Plasma Concentration Prediction', 'Parameters')
)

# Trace 1: Data Points (3D scatter)
fig.add_trace(go.Scatter3d(
    x=dose_mg, y=egfr, z=peak_concentration,
    mode='markers',
    marker=dict(size=5, color='blue', opacity=0.7),
    name='Patients'
), row=1, col=1)

# Trace 2: Fitted Plane
fig.add_trace(go.Surface(
    x=Dose_mesh, y=Egfr_mesh, z=Conc_plane,
    colorscale=[[0, plane_color], [1, plane_color]],
    opacity=plane_opacity,
    showscale=False,
    name='Fitted Plane'
), row=1, col=1)

# Trace 3: Parameter bar chart
param_names = ['Dose (/100mg)', 'eGFR (/10)', 'Intercept']
param_values = [w1_val, w2_val, b_val]
colors = ['green' if v >= 0 else 'red' for v in param_values]
fig.add_trace(go.Bar(
    x=param_names,
    y=param_values,
    marker_color=colors,
    text=[f'{v:.2f}' for v in param_values],
    textposition='auto',
    name='Parameters'
), row=1, col=2)

# Camera setup
camera = dict(eye=dict(x=1.5, y=1.5, z=1.2))

fig.update_layout(
    title=f'State: {status_msg} | Loss: {current_loss:.2f}',
    scene=dict(
        xaxis_title='Dose (mg)',
        yaxis_title='eGFR (mL/min/1.73m²)',
        zaxis_title='Peak Concentration (mg/L)',
        camera=camera
    ),
    width=1100, height=600,
    showlegend=False,
    margin=dict(l=0, r=0, b=0, t=60)
)

# Fix parameter bar chart y-axis range
fig.update_yaxes(range=[-2, 8], row=1, col=2)

fig.show()

print(f"Model: Cmax = {w1_val:.4f}×Dose(/100mg) + {w2_val:.4f}×eGFR(/10) + {b_val:.4f}")
print(f"True:  Cmax = {true_w1}×Dose(/100mg) + {true_w2}×eGFR(/10) + {true_b}")


Model: Cmax = 2.1544×Dose(/100mg) + -0.4089×eGFR(/10) + 4.9075
True:  Cmax = 2.2×Dose(/100mg) + -0.45×eGFR(/10) + 5.0
